In [ ]:
import pandas as pd
import os
import numpy as np
DATA_DIR = '/path/to/CausalFair/Resume/job-distribution'

df_ipums = pd.read_csv(os.path.join(DATA_DIR, 'processed_job_data_0102.csv'))
df_ipums.rename(columns={
    'SEX_GROUP': 'sex',
    'AGE': 'age',
    'RACE_GROUP': 'race',
    'EDU_LEVEL': 'edu_level',
    'STATE_NAME': 'state_name',
    'job': 'job',
}, inplace=True)

df_psid = pd.read_csv(os.path.join(DATA_DIR + '/sources/PSID', 'psid_famind_final_filtered_0121.csv'))

JOBS = df_ipums['job'].unique()
# Race mismatch

df_ipums = df_ipums[df_ipums['race'].isin(['White', 'Black', 'Asian/Pacific Islander', 'AIAN'])]
df_psid = df_psid[df_psid['race'].isin(['White', 'Black', 'Asian/Pacific Islander', 'AIAN'])]

# Edu level mismatch
df_ipums = df_ipums[df_ipums['edu_level'].isin(["Associate's degree", "Bachelor's", "Doctorate", "High school graduate/GED", "Master's", "Professional degree", "Some college (no degree)"])]
df_psid = df_psid[df_psid['edu_level'].isin(["Associate's degree", "Bachelor's", "Doctorate", "High school graduate/GED", "Master's", "Professional degree", "Some college (no degree)"])]

# experience year na
df_psid = df_psid[df_psid['total_experience_years'].notna()]
df_psid = df_psid[df_psid['total_experience_years'] > 0]

df_ipums['sex'] = df_ipums['sex'].map({
    'Female': 0,
    'Male': 1,
})

df_psid['sex'] = df_psid['sex'].map({
    1: 1,
    2: 0,
})

df_psid_original = df_psid.copy()

# Age common support

AGE_CUTOFF = 44
df_ipums = df_ipums[(df_ipums['age'] >= 18) & (df_ipums['age'] <= AGE_CUTOFF)]
df_psid = df_psid[(df_psid['age'] >= 18) & (df_psid['age'] <= AGE_CUTOFF)]

# major_major for ipums|
df_ipums['major_major_occupation_group'] = df_ipums['job'].map({
    'software_developers': 'Professional and Related',
    'elementary_middle_school_teachers': 'Professional and Related',
    'registered_nurses': 'Professional and Related',
    'accountants_auditors': 'Management, Business, and Financial',
    'construction_laborers': 'Natural Resources, Construction, and Maintenance'
})

# df_psid_person_individual_id

df_psid['id'] = df_psid['person_type'] + '_' + df_psid['individual_id'].astype(str)
# filter unique id
df_psid = df_psid[~df_psid['id'].duplicated()]


print(df_psid.shape)
# print(df_psid['job'].value_counts())
print(df_psid['major_major_occupation_group'].value_counts())


In [ ]:
df_ipums['job'].value_counts()

In [ ]:
df_ipums['survey_year'] = 2023

In [ ]:
MINIMUM_WORK_AGE = 17

df_psid['r'] = df_psid['total_experience_years'] / (df_psid['age'] - MINIMUM_WORK_AGE)
df_psid = df_psid[(df_psid['r'] <= 1) | (df_psid['age'] == 18)]

In [ ]:
IPUMS_COLUMNS = ['sex', 'age', 'race', 'edu_level', 'state_name', 'survey_year', 'job']
PSID_COLUMNS = ['sex', 'age', 'race',  'edu_level', 'state_name', 'survey_year','total_experience_years', 'r']


MATCHING_FEATURES = ['sex', 'age', 'race', 'edu_level', 'state_name']                                                         
# df_ipums = df_ipums[IPUMS_COLUMNS]
# df_psid = df_psid[PSID_COLUMNS]
# print(df_ipums.shape, df_psid.shape)

df_ipums_merge = df_ipums[MATCHING_FEATURES]
df_psid_merge = df_psid[MATCHING_FEATURES]

df_merge = pd.concat([df_ipums_merge, df_psid_merge], axis=0)

In [ ]:
X_all = pd.get_dummies(df_merge[MATCHING_FEATURES], columns=['race', 'edu_level', 'state_name'])

In [ ]:
domain = np.concatenate([
    np.ones(len(df_ipums)),
    np.zeros(len(df_psid)),
])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ("lr", LogisticRegression(
        penalty="l2",
        C=5e-5,                 # smaller = weaker regularization (0.01-0.5 recommended)
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced"
    ))
])

In [ ]:
clf.fit(X_all, domain)

In [ ]:
# clf accuracy
clf.score(X_all, domain)

In [ ]:
X1 = pd.get_dummies(df_ipums[['sex', 'age', 'race', 'edu_level', 'state_name']], columns=['race', 'edu_level', 'state_name'])
X2 = pd.get_dummies(df_psid[['sex', 'age', 'race', 'edu_level', 'state_name']], columns=['race', 'edu_level', 'state_name'])


p = clf.predict_proba(X2)[:, 1]


In [ ]:
print(X1.shape, X2.shape)

In [ ]:
# pi1 = len(X1) / len(X_all)
# pi2 = len(X2) / len(X_all)

w = (p / (1 - p)) 



In [ ]:
import matplotlib.pyplot as plt
plt.hist(w, bins=100)
plt.yscale("log")
plt.title("Importance weights (XGBoost)")
plt.show()

In [ ]:
q = np.quantile(w, 0.99)
w_clipped = np.clip(w, 0, q)
print(q)

In [ ]:
ESS = (w_clipped.sum() ** 2) / (np.sum(w_clipped ** 2))
ESS_ratio = ESS / len(w_clipped)
print(ESS_ratio)

In [ ]:
df_psid['w'] = w

In [ ]:
import os
import json
from random import sample
import numpy as np
import pandas as pd
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import make_scorer

# use cupy when a GPU is available
try:
    import cupy as cp
    USE_GPU = True
    print("Using GPU (cupy)")
except ImportError:
    cp = np
    USE_GPU = False
    print("Using CPU (numpy)")


def make_constrained_exp_mse_scorer(age_col_idx, minimum_work_age=17):
    """
    MSE scorer for direct prediction of experience years.
    Clips predictions to [0, age - minimum_work_age] before computing the MSE.
    
    Parameters:
    -----------
    age_col_idx : int
        index of the age column in X
    minimum_work_age : int
        minimum working age (default: 18)
    """
    def constrained_exp_mse_scorer(estimator, X_val, exp_true, sample_weight=None):
        # take .values from a DataFrame, otherwise use as-is
        X_arr = np.asarray(X_val)
        exp_true = np.asarray(exp_true)
        
        # predict on the original X_val so feature names are preserved
        exp_pred = estimator.predict(X_val)
        
        # pull out age and compute max_possible_exp (as a numpy array)
        age = X_arr[:, age_col_idx]
        max_possible_exp = np.maximum(0, age - minimum_work_age)
        
        # apply the constraint: clip predictions to [0, max_possible_exp]
        exp_pred_clipped = np.clip(exp_pred, 0, max_possible_exp)
        
        # compute the MSE
        squared_errors = (exp_true - exp_pred_clipped) ** 2
        
        if sample_weight is not None:
            mse = np.average(squared_errors, weights=sample_weight)
        else:
            mse = np.mean(squared_errors)
        
        return -mse  # sklearn maximizes the score, so return the negative
    
    return constrained_exp_mse_scorer


def fit(X, Y, binary=False, categorical=False, **kwargs):
    X = cp.asarray(X)
    Y = cp.asarray(Y)
    model_class = xgb.XGBClassifier if binary or categorical else xgb.XGBRegressor
    
    device_params = {'tree_method': 'hist', 'device': 'cuda:0'}
    m = model_class(
        **device_params,
        **kwargs
    ).fit(X, Y)
    return m


def pred(X, m, binary=True, clip_val=0.0, **kwargs):
    X = cp.asarray(X)
    if binary:
        y = m.predict_proba(X)[:, 1]
    else:
        y = m.predict(X)
    if clip_val:
        y = np.clip(y, a_min=clip_val, a_max=1-clip_val)
    return y


def grid_search(X, Y, binary, categorical, hparams, sample_weight=None):
    model_class = xgb.XGBClassifier if binary or categorical else xgb.XGBRegressor
    # scoring = 'neg_brier_score' if binary or categorical else 'neg_mean_squared_error'
    scoring = make_constrained_exp_mse_scorer(1, 17)
    m = model_class(
        tree_method='hist',
        device='cuda:0',
        verbosity=0,
    )
    
    cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
    )

    grid_search = GridSearchCV(
        estimator=m, param_grid=hparams,
        scoring=scoring, cv=cv, verbose=0, 
    )
    grid_search.fit(X, Y, sample_weight=sample_weight)
    return grid_search

# p(r | sex, age, race, edu_level, state_name, major_major_occupation_group), with density ratio based weighting

In [ ]:

df_job = df_psid.copy()

# Feature matrix
X = pd.get_dummies(
    df_job[['sex', 'age', 'race', 'edu_level', 'state_name', 'major_major_occupation_group', 'survey_year']], 
    columns=['race', 'edu_level', 'state_name', 'major_major_occupation_group', 'survey_year']
)
X = X.astype(np.float32)

# Target: total_experience_years (predicted directly)
Y = df_job['total_experience_years'].values

# locate the age column index
age_col_idx = X.columns.get_loc('age')
print(f"Age column index: {age_col_idx}")

# Hyperparameter grid
HPARAMS = {
    'n_estimators': [100],
    'max_depth': [3],
    'reg_lambda': [100],
    'min_child_weight': [1],
    'colsample_bytree': [1.0],
}

# Grid search
# takes around 2min for training with best hyperparams
grid = grid_search(
    X, df_job['total_experience_years'],
    binary=False,
    categorical=False,
    hparams=HPARAMS,
    sample_weight=df_job['w'].values
)

# inspect the result
results = pd.DataFrame(grid.cv_results_)
results['mse'] = -results['mean_test_score']
print(results[['params', 'mse', 'std_test_score']].sort_values(by='mse').head(10))

In [ ]:
print(f"Best params: {grid.best_params_}")

# predict on PSID and evaluate
exp_pred_raw = grid.predict(X)

# apply the constraint
max_possible_exp = df_psid['age'].values - MINIMUM_WORK_AGE
exp_pred = np.clip(exp_pred_raw, 0, max_possible_exp)

df_psid['pred_exp'] = exp_pred
df_psid['pred_exp_raw'] = exp_pred_raw  # before the constraint is applied

# R^2
r2 = df_psid['pred_exp'].corr(df_psid['total_experience_years'])**2
print(f"R^2: {r2:.4f}")

# MSE
mse = ((df_psid['total_experience_years'] - df_psid['pred_exp'])**2).mean()
print(f"MSE: {mse:.4f}")

# check for constraint violations
n_violations = ((exp_pred_raw < 0) | (exp_pred_raw > max_possible_exp)).sum()
print(f"Constraint violations (before clipping): {n_violations} ({100*n_violations/len(exp_pred_raw):.2f}%)")

# Sampling

In [ ]:
from scipy.stats import truncnorm
from tqdm import tqdm

# compute residuals (on PSID)
df_psid['residual'] = df_psid['total_experience_years'] - df_psid['pred_exp']

# define the age bins
age_bins = [17, 25, 35, AGE_CUTOFF + 1]
df_psid['age_bin'] = pd.cut(df_psid['age'], bins=age_bins, right=False)

# sigma per Age x Gender x Education Level
sigma_by_group = (
    df_psid
    .groupby(['age_bin', 'sex', 'major_major_occupation_group'])['residual']
    .std()
    .to_dict()
)

print("Sigma by (age_bin, sex, major_major_occupation_group):")
for k, v in sorted(sigma_by_group.items(), key=lambda x: (str(x[0][0]), x[0][1])):
    print(f"  {k}: {v:.3f}" if pd.notna(v) else f"  {k}: NaN")

# fallback sigmas, used when a group has no data
sigma_by_age_sex = df_psid.groupby(['age_bin', 'sex'])['residual'].std().to_dict()
sigma_by_age = df_psid.groupby('age_bin')['residual'].std().to_dict()
global_sigma = df_psid['residual'].std()
print(f"\nGlobal sigma (fallback): {global_sigma:.3f}")


# prepare the IPUMS data
df_ipums['age_bin'] = pd.cut(df_ipums['age'], bins=age_bins, right=False)
df_ipums['max_possible_exp'] = df_ipums['age'] - MINIMUM_WORK_AGE

# Feature matrix for IPUMS
X_ipums = pd.get_dummies(
    df_ipums[['sex', 'age', 'race', 'edu_level', 'state_name', 'major_major_occupation_group', 'survey_year']],
    columns=['race', 'edu_level', 'state_name', 'major_major_occupation_group', 'survey_year']
)

X_ipums = X_ipums.reindex(columns=X.columns, fill_value=0)                                                                                                    
X_ipums = X_ipums.astype(np.float32)

# predict
exp_hat = grid.predict(X_ipums)
exp_hat = np.clip(exp_hat, 0, df_ipums['age'] - MINIMUM_WORK_AGE).values

print(f"IPUMS shape: {X_ipums.shape}")
print(f"Predicted exp range: [{exp_hat.min():.2f}, {exp_hat.max():.2f}]")


# truncated-normal sampling with a per Age x Gender x Education Level sigma

def get_sigma(age_bin, sex, major_major_occupation_group):
    """Return sigma via a hierarchical fallback."""
    # 1st choice: age_bin x sex x edu_level
    sigma = sigma_by_group.get((age_bin, sex, major_major_occupation_group), np.nan)
    if pd.notna(sigma) and sigma > 0:
        return sigma
    
    # 2nd choice: age_bin x sex
    sigma = sigma_by_age_sex.get((age_bin, sex), np.nan)
    if pd.notna(sigma) and sigma > 0:
        return sigma
    
    # 3rd choice: age_bin
    sigma = sigma_by_age.get(age_bin, np.nan)
    if pd.notna(sigma) and sigma > 0:
        return sigma
    
    # 4th choice: global
    return global_sigma


exp_sampled = np.empty(len(X_ipums))

for i in tqdm(range(len(X_ipums))):
    mu = exp_hat[i]
    row = df_ipums.iloc[i]
    age = row['age']
    max_exp = age - MINIMUM_WORK_AGE
    
    # sigma per age_bin x sex x edu_level (hierarchical fallback)
    age_bin = row['age_bin']
    sex = row['sex']

    major_major_occupation_group = row['major_major_occupation_group']
    sigma = get_sigma(age_bin, sex, major_major_occupation_group)
    
    # numerical safety
    sigma = max(sigma, 1e-4)
    
    # truncated normal parameters
    # bounds: [0, max_exp] (experience is >= 0 and <= age - 18)
    a = (0.0 - mu) / sigma
    b = (max_exp - mu) / sigma
    

    exp_sampled[i] = truncnorm.rvs(a, b, loc=mu, scale=sigma)
    
# assign the results
X_ipums['pred_exp_raw'] = exp_hat  # raw model prediction
X_ipums['pred_exp'] = exp_sampled  # sampled value (constrained)

# safety clip (unnecessary in theory, but recommended in practice)
X_ipums['pred_exp'] = np.clip(
    X_ipums['pred_exp'],
    0,
    df_ipums['max_possible_exp']
)

# verify the constraint
max_possible = df_ipums['max_possible_exp'].values
constraint_satisfied = (X_ipums['pred_exp'] >= 0) & (X_ipums['pred_exp'] <= max_possible)
print(f"Constraint satisfied: {constraint_satisfied.sum()} / {len(X_ipums)} ({100*constraint_satisfied.mean():.2f}%)")

# basic statistics
print(f"\nSampled exp statistics:")
print(f"  Min: {X_ipums['pred_exp'].min():.2f}")
print(f"  Max: {X_ipums['pred_exp'].max():.2f}")
print(f"  Mean: {X_ipums['pred_exp'].mean():.2f}")
print(f"  Std: {X_ipums['pred_exp'].std():.2f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Age vs Predicted Exp (hexbin)
ax1 = axes[0]
hb = ax1.hexbin(
    df_ipums['age'].values,
    exp_sampled,
    gridsize=50,
    cmap='Blues',
    mincnt=1,

)
ax1.plot([18, AGE_CUTOFF+1], [0, AGE_CUTOFF-MINIMUM_WORK_AGE], 'r--', label='Max possible (age - 17)', alpha=0.7)
ax1.set_xlabel('Age')
ax1.set_ylabel('Predicted Experience Years')
ax1.set_title('Age vs Sampled Experience')
ax1.legend()
plt.colorbar(hb, ax=ax1)

# 2. Distribution of sampled exp
ax2 = axes[1]
ax2.hist(exp_sampled, bins=50, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Experience Years')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Sampled Experience')

# 3. Raw prediction vs Sampled (scatter sample)
ax3 = axes[2]
sample_idx = np.random.choice(len(exp_hat), min(5000, len(exp_hat)), replace=False)
ax3.scatter(exp_hat[sample_idx], exp_sampled[sample_idx], alpha=0.3, s=5)
ax3.plot([0, AGE_CUTOFF-MINIMUM_WORK_AGE], [0, AGE_CUTOFF-MINIMUM_WORK_AGE], 'r--', label='y=x')
ax3.set_xlabel('Raw Prediction')
ax3.set_ylabel('Sampled (Constrained)')
ax3.set_title('Raw vs Constrained Prediction')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
X_ipums_final = df_ipums.copy()
df_ipums['pred_exp'] = X_ipums['pred_exp'].values
# save to csv
df_ipums.to_csv(os.path.join(DATA_DIR, 'processed_job_data_0102_with_exp_pred.csv'), index=False)